In [25]:
# 모듈 import
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.utils import plot_model

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.layers import Concatenate, Dropout
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
from sklearn.preprocessing import LabelEncoder

In [48]:
# 이미지 갖고오기
df = pd.read_csv('data/cifar10/trainLabels.csv')
image_dir = './data/cifar10/train/train/'

# x데이터랑 t데이터 매칭
x_data = [os.path.join(image_dir, str(fname) + '.png') for fname in df['id']]
# t데이터 숫자로 인코딩
le = LabelEncoder()
t_data = le.fit_transform(df['label'])

In [49]:
# 데이터 확인
print(x_data[:5])
print(t_data[:7])
df['id'] = df['id'].astype(str)+ '.png'
print(df['id'].head())

['./data/cifar10/train/train/1.png', './data/cifar10/train/train/2.png', './data/cifar10/train/train/3.png', './data/cifar10/train/train/4.png', './data/cifar10/train/train/5.png']
[6 9 9 4 1 1 2]
0    1.png
1    2.png
2    3.png
3    4.png
4    5.png
Name: id, dtype: object


In [50]:
# Parameter 설정
IMAGE_SIZE = 224
BATCH_SIZE = 32

In [51]:
# 학습용과 테스트용으로 분리
x_data_train, x_data_test, t_data_train, t_data_test = train_test_split(x_data,
                                                                       t_data,
                                                                       test_size=0.2,
                                                                       stratify=t_data)

In [52]:
# 이미지 불러오고 전처리까지 하는 함수
def parse_image(filename, label):
    image = tf.io.read_file(filename)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE,IMAGE_SIZE])
    image = tf.keras.applications.efficientnet.preprocess_input(image)
    return image, label

# 학습 데이터만 증강하는 함수
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    return image, label
    
# 데이터셋 생성 함수
def make_dataset(x,t,train=False):
    dataset = tf.data.Dataset.from_tensor_slices((x,t))
    dataset = dataset.map(parse_image,
                          num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        dataset = dataset.shuffle(buffer_size=1000)
        dataset = dataset.map(augment,
                              num_parallel_calls=tf.data.AUTOTUNE)    
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [53]:
train_dataset = make_dataset(x_data_train, t_data_train, train=True)
validation_dataset = make_dataset(x_data_test, t_data_test, train=False)

In [54]:
# model
model_base = EfficientNetB0(weights='imagenet',
                            include_top=False,
                            input_shape=(IMAGE_SIZE,IMAGE_SIZE,3))
for layer in model_base.layers:
    layer.trainable = False

In [55]:
inputs = Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = model_base(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(64)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(10, activation='softmax')(x)

model = Model(inputs, outputs)

In [56]:
# model 설정
model.compile(optimizer=Adam(learning_rate=1e-3),
             loss='sparse_categorical_crossentropy',
             metrics=['accuracy'])

In [57]:
es_callback = EarlyStopping(monitor='val_loss',
                           patience=5,
                           restore_best_weights=True,
                           verbose=1)
cp_callback = ModelCheckpoint(filepath='./efficientnetb0_weights.h5',
                             save_best_only=True,
                             save_weights_only=True,
                             monitor='val_accuracy',
                             verbose=1)

In [58]:
# 1차 학습 진행
model.fit(train_dataset,
         epochs=20,          
         validation_data=validation_dataset,
         callbacks=[es_callback, cp_callback],
         verbose=1)

Epoch 1/20
1249/1250 [============================>.] - ETA: 0s - loss: 0.4851 - accuracy: 0.8462     
Epoch 1: val_accuracy improved from -inf to 0.89880, saving model to ./efficientnetb0_weights.h5
1250/1250 [==============================] - 57s 42ms/step - loss: 0.4850 - accuracy: 0.8462 - val_loss: 0.2965 - val_accuracy: 0.8988
Epoch 2/20
1249/1250 [============================>.] - ETA: 0s - loss: 0.3418 - accuracy: 0.8870  
Epoch 2: val_accuracy improved from 0.89880 to 0.90550, saving model to ./efficientnetb0_weights.h5
1250/1250 [==============================] - 51s 41ms/step - loss: 0.3416 - accuracy: 0.8870 - val_loss: 0.2698 - val_accuracy: 0.9055
Epoch 3/20
1249/1250 [============================>.] - ETA: 0s - loss: 0.3027 - accuracy: 0.8989  
Epoch 3: val_accuracy improved from 0.90550 to 0.90750, saving model to ./efficientnetb0_weights.h5
1250/1250 [==============================] - 51s 41ms/step - loss: 0.3028 - accuracy: 0.8989 - val_loss: 0.2646 - val_accuracy: 0.

In [59]:
# Fine Tuning
model_base.trainable = True

for layer in model_base.layers[:-30]:
    layer.trainable = False

In [60]:
# model 재설정
model.compile(optimizer=Adam(learning_rate=1e-5),
             loss='sparse_categorical_crossentropy',
             metrics=['accuracy'])

In [61]:
# model 재학습
model.fit(train_dataset,
         epochs=25,          
         validation_data=validation_dataset,
         callbacks=[es_callback, cp_callback],
         verbose=1)

Epoch 1/25
1249/1250 [============================>.] - ETA: 0s - loss: 0.2139 - accuracy: 0.9261     
Epoch 1: val_accuracy improved from 0.91450 to 0.92170, saving model to ./efficientnetb0_weights.h5
1250/1250 [==============================] - 66s 48ms/step - loss: 0.2138 - accuracy: 0.9261 - val_loss: 0.2302 - val_accuracy: 0.9217
Epoch 2/25
1249/1250 [============================>.] - ETA: 0s - loss: 0.1893 - accuracy: 0.9364  
Epoch 2: val_accuracy improved from 0.92170 to 0.92490, saving model to ./efficientnetb0_weights.h5
1250/1250 [==============================] - 59s 47ms/step - loss: 0.1892 - accuracy: 0.9364 - val_loss: 0.2214 - val_accuracy: 0.9249
Epoch 3/25
1250/1250 [==============================] - ETA: 0s - loss: 0.1761 - accuracy: 0.9404  
Epoch 3: val_accuracy improved from 0.92490 to 0.92580, saving model to ./efficientnetb0_weights.h5
1250/1250 [==============================] - 60s 48ms/step - loss: 0.1761 - accuracy: 0.9404 - val_loss: 0.2166 - val_accuracy:

In [68]:
# 테스트 제출 파일 만들기
# 1. submission 파일 먼저 불러오기 (test 이미지 id가 들어 있음)
submission = pd.read_csv('./data/cifar10/sampleSubmission.csv')  # 총 300,000행
submission['id'] = submission['id'].astype(str)+ '.png'  # 정수 → 문자열
print(submission['id'].head())

0    1.png
1    2.png
2    3.png
3    4.png
4    5.png
Name: id, dtype: object


In [73]:
# 테스트 이미지 경로 생성
image_dir = './data/cifar10/test/test'  # 실제 테스트 이미지들이 있는 경로
image_paths = [os.path.join(image_dir, fname) for fname in submission['id']]  
# 이미 .png가 포함된 상태로 경로 생성


In [74]:
# 테스트 데이터셋 생성 함수
def make_test_dataset(x):
    dummy_labels = [0] * len(x)  # 임시 라벨 (실제 사용 안 함)
    dataset = tf.data.Dataset.from_tensor_slices((x,dummy_labels))
    dataset = dataset.map(parse_image,
                          num_parallel_calls=tf.data.AUTOTUNE)   
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [75]:
test_dataset = make_test_dataset(image_paths)

In [77]:
# 3. 예측 수행
preds = model.predict(test_dataset, verbose=1)
pred_labels = np.argmax(preds, axis=1)  # 예측 확률 → 클래스 index

9375/9375 [==============================] - 275s 29ms/step


In [78]:
# 4. 클래스 index → 클래스 이름 (LabelEncoder 사용해서 inverse_transform)
# 훈련 때 LabelEncoder를 le로 정의했음 (le.classes_ 순서로 인코딩)
pred_label_names = le.inverse_transform(pred_labels)

In [79]:
# 5. submission 파일에 결과 삽입
submission['label'] = pred_label_names

In [80]:
# 6. id에서 .png 제거
submission['id'] = submission['id'].str.replace('.png', '', regex=False)
print(submission['id'].head())

0    1
1    2
2    3
3    4
4    5
Name: id, dtype: object


In [81]:
# 7. 제출 파일 저장
submission.to_csv('submission_cifar10_dataset.csv', index=False)